# Prepare an account sample — run before the baseline

**Default: 500 accounts, seed 42, all linked tables and every raw transaction for those accounts.** The input is full raw Berka data, not the earlier 20,000-row transaction sample.

1. Restore the `.venv` environment: `uv sync --locked --group tabular-data --inexact`.
2. Review the paths/count below and preview the cohort on CPU.
3. Set `PREPARE_DATASET = True` yourself to write the separate processed dataset.
4. Open [00_baseline.ipynb](00_baseline.ipynb) and establish its matching baseline.

Change [dataset_settings.py](dataset_settings.py) once to select another dataset for all notebooks. Existing data, checkpoints, and synthesized files are preserved. You execute this notebook yourself.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display as show


ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from implementations.tabular_data.multi_table.data_preprocessing.account_sampling import (
    load_account_sample,
    prepare_account_sample,
)
from implementations.tabular_data.multi_table.experiments.dataset_settings import (
    ACCOUNT_SAMPLE_SIZE as DEFAULT_ACCOUNT_SAMPLE_SIZE,
)
from implementations.tabular_data.multi_table.experiments.dataset_settings import (
    DATA_DIR as DEFAULT_DATA_DIR,
)
from implementations.tabular_data.multi_table.experiments.dataset_settings import (
    DATA_SEED as DEFAULT_DATA_SEED,
)
from implementations.tabular_data.multi_table.experiments.dataset_settings import (
    RAW_DATA_DIR as DEFAULT_RAW_DATA_DIR,
)


## 1. Choose the raw data, output path, and sample

The shared defaults are **500 accounts / seed 42**. Edit `dataset_settings.py` to change those values or select different raw/processed paths, then restart notebook kernels. `ACCOUNT_SAMPLE_SIZE = None` selects all accounts.

You may override the variables here. If doing so, set a new `OUTPUT_DATA_DIR` and point the shared `DATA_DIR` at it before the baseline/experiments. Raw input must contain full Berka transactions and the other 7 tables. The helper checks every required file; it does not download or overwrite source data.

In [2]:
RAW_DATA_DIR = DEFAULT_RAW_DATA_DIR
ACCOUNT_SAMPLE_SIZE = DEFAULT_ACCOUNT_SAMPLE_SIZE  # 500 by default; None means all.
SAMPLING_SEED = DEFAULT_DATA_SEED  # 42 by default.
OUTPUT_DATA_DIR = DEFAULT_DATA_DIR
PREPARE_DATASET = True
print(f"Full raw tables: {RAW_DATA_DIR}")
print(f"Separate processed dataset: {OUTPUT_DATA_DIR}")


Full raw tables: /home/coder/synthetic-data-bootcamp/implementations/tabular_data/multi_table/data/berka/raw_data
Separate processed dataset: /home/coder/synthetic-data-bootcamp/implementations/tabular_data/multi_table/data/berka_accounts_500_seed42


## 2. Preview the cohort on CPU

Account IDs are selected uniformly with a fixed seed. Keep their transactions, loans, orders, and dispositions; cards follow retained dispositions; clients follow retained dispositions; districts are the union referenced by accounts and clients. A shared client does not pull unselected accounts into the cohort.

This preserves complete **observed raw** histories for selected accounts and valid links. It does not guarantee that the generator will preserve chronology. There is no target-based oversampling and no transaction-row limit. Inspect event and small-table counts before training; larger cohorts require more generation time.

In [3]:
SAMPLED_TABLES, SAMPLE_INFO = load_account_sample(RAW_DATA_DIR, ACCOUNT_SAMPLE_SIZE, seed=SAMPLING_SEED)
show(pd.DataFrame({"raw_rows": SAMPLE_INFO["source_n_rows"], "selected_rows": SAMPLE_INFO["n_rows"]}))
show(pd.Series({key: SAMPLE_INFO[key] for key in ["account_sample_size", "seed", "overdraft_rows",
                                                "overdraft_accounts", "overdraft_prevalence"]}))
small = [name for name, frame in SAMPLED_TABLES.items() if len(frame) < 100]
print(f"Tables with fewer than 100 rows: {small}. Interpret clustering/training results carefully.")
print("Preview only; no dataset files have been written by this cell.")


,raw_rows,selected_rows
account,4500,500
card,892,90
client,5369,596
disp,5369,596
district,77,77
loan,682,76
order,6471,742
trans,1056320,117560


account_sample_size     500.000000
seed                     42.000000
overdraft_rows          247.000000
overdraft_accounts       28.000000
overdraft_prevalence      0.002101
dtype: float64

Tables with fewer than 100 rows: ['loan', 'card', 'district']. Interpret clustering/training results carefully.
Preview only; no dataset files have been written by this cell.


## 3. Write the processed dataset

Enable `PREPARE_DATASET` in step 1 yourself. The same existing preprocessing transforms are applied to the selected raw rows. Selected IDs, source/config fingerprints, per-table counts, date epochs, codebooks, and artifact checksums are saved.

An identical completed dataset can be reused after verification. An existing unrelated, changed, failed, or differently configured output is rejected; select a new output path rather than replacing files. This step uses CPU and performs no model training or synthetic generation.

In [4]:
PREPARATION = None
if PREPARE_DATASET:
    PREPARATION = prepare_account_sample(RAW_DATA_DIR, OUTPUT_DATA_DIR, ACCOUNT_SAMPLE_SIZE, seed=SAMPLING_SEED)
    print(f"Dataset ready: {PREPARATION['output_dir']}; reused={PREPARATION['reused']}")
    print("Next: select this DATA_DIR and run 00_baseline.ipynb.")
else:
    print("Preview complete. Set PREPARE_DATASET = True to write the selected cohort.")


Preprocessing supplied raw account ...
  500 rows, columns=['account_id', 'district_id', 'frequency', 'date']
  Wrote /home/coder/synthetic-data-bootcamp/implementations/tabular_data/multi_table/data/berka_accounts_500_seed42/account.csv
  Wrote /home/coder/synthetic-data-bootcamp/implementations/tabular_data/multi_table/data/berka_accounts_500_seed42/account_domain.json
  LabelEncoder classes:
    frequency: ['POPLATEK MESICNE', 'POPLATEK PO OBRATU', 'POPLATEK TYDNE']
  Date epoch (YYMMDD): 930103
Preprocessing supplied raw card ...
  90 rows, columns=['card_id', 'disp_id', 'type', 'issued']
  Wrote /home/coder/synthetic-data-bootcamp/implementations/tabular_data/multi_table/data/berka_accounts_500_seed42/card.csv
  Wrote /home/coder/synthetic-data-bootcamp/implementations/tabular_data/multi_table/data/berka_accounts_500_seed42/card_domain.json
  LabelEncoder classes:
    card_type: ['classic', 'gold', 'junior']
  Date epoch (YYMMDD): 940405
Preprocessing supplied raw client ...
  596

## 4. Changing datasets later

Update `ACCOUNT_SAMPLE_SIZE` / `DATA_SEED` in [dataset_settings.py](dataset_settings.py) to derive a new default folder, or set `DATA_DIR` to an existing processed Berka-compatible dataset. If the raw source changes, update `RAW_DATA_DIR` too. Restart kernels after editing shared settings.

Keep all 8 CSVs/domains plus `dataset_meta.json` and `preprocess_meta.json`. All experiment notebooks expose `DATA_DIR = DEFAULT_DATA_DIR`, so the selected shared path is visible and can be overridden. Data fingerprints keep different contents in separate model caches and reports even when using the same experiment root.

Establish a matching baseline for every changed dataset. The evaluator rejects a trial whose recorded input fingerprint differs from the selected reference. Existing synthesized artifacts are retained.